# 30. `model_with_fitted_values` and `FitSession.fit(update_model=True)`

**Objectives:**
- Fit a small toy model and show that `session.model.parameters` still holds the
  *initial* values after `fit()` -- `DecayModel`/`Parameter` are frozen dataclasses,
  and fitting never mutates them in place; this is documented behavior, not a bug.
- Call `session.fit(update_model=True)` and show the returned `updated_model`'s
  parameters now hold the fitted values.
- Export `updated_model` with `export_model`, i.e. a frozen, re-loadable snapshot of
  the fit result.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant, Parameter, RealImag,
    Resonance, export_model, generate_toy,
)

## 1. Build and fit a small toy model

`rho`'s coefficient is a fixed `RealImag(1.0, 0.0)` reference (an overall complex
rescaling of every component leaves the normalized intensity unchanged, so one
component's coefficient must stay fixed for the rest to be identifiable). `NR`'s
coefficient is built with `Parameter.coefficient(..., owner=...)` so
`FitSession.fit(start_values=...)` can float it -- a plain `RealImag` of bare floats
cannot be floated.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

nr_coefficient = RealImag(
    Parameter.coefficient("NR.x", 0.3, owner="NR", bounds=(-3.0, 3.0)),
    Parameter.coefficient("NR.y", -0.15, owner="NR", bounds=(-3.0, 3.0)),
)
model = DecayModel(
    channel,
    [Resonance("rho", pair=(0, 1), coefficient=RealImag(1.0, 0.0),
               mass=0.77526, width=0.1491, spin=1),
     NonResonant(nr_coefficient, name="NR")],
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=40,
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(
    model, 1000, parameters=truth, seed=30,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)

session = FitSession(model, data)
start = {"NR.x": 0.5, "NR.y": -0.3}
result = session.fit(start, simplex=True, ncall=5000)
assert result.valid
print("Fitted values:", session.result_values(result))

Fitted values: {'NR.x': 0.32934741835929493, 'NR.y': -0.09022913698604212}


## 2. `session.model` is unchanged after `fit()`

`fit()` reports the best-fit values in a separate `result` object
(`session.result_values(result)`); `session.model` -- and every `Parameter` inside
it -- is the exact same frozen object passed into `FitSession`, still holding the
start values it was built with.

In [3]:
still_initial = {p.name: p.value for p in session.model.parameters}
print("session.model.parameters (unchanged):", still_initial)
assert still_initial == truth
print("session.model still holds the initial values, as documented.")

session.model.parameters (unchanged): {'NR.x': 0.3, 'NR.y': -0.15}
session.model still holds the initial values, as documented.


## 3. `update_model=True` bakes the fit result into a new model

Passing `update_model=True` changes `fit()`'s return value to `(result, updated_model)`.
`updated_model` is a fresh `DecayModel` (built via `model_with_fitted_values`) whose
floated parameters' `.value` are set to their fitted results; fixed parameters are
unchanged.

In [4]:
result2, updated_model = session.fit(start, simplex=True, ncall=5000, update_model=True)
assert result2.valid

updated_values = {p.name: p.value for p in updated_model.parameters}
fitted_values = session.result_values(result2)
print("updated_model.parameters:", updated_values)
assert updated_values == fitted_values
print("updated_model now holds the fitted values.")

updated_model.parameters: {'NR.x': 0.32934741835929493, 'NR.y': -0.09022913698604212}
updated_model now holds the fitted values.


## 4. Export the fitted snapshot

`export_model` on `updated_model` writes out the *fitted* values, unlike exporting
`model` (or `session`) directly right after a fit, which would still write the initial
start values.

In [5]:
export_model(updated_model, "tutorial_30_model.json")
print("Exported fitted snapshot to tutorial_30_model.json")

Exported fitted snapshot to tutorial_30_model.json


## Continue learning

See [docs/model_io.md](../../docs/model_io.md), "Exporting a model right after a fit",
and [tutorial 28](tutorial_28_export_import_model.ipynb) for the full export/import
round trip. Return to [the course guide](TUTORIALS.md).